# Explore the DBP API

**Sign in → inspect metrics → filter videos → assign subjects → download → track presentation.**

The website stores the catalog and assignments. This client fetches data; PGL or PsychoPy plays it. Run one cell at a time. Creation and downloads below are explicit; previewing never records a trial.

## API objects at a glance

Main objects and selected methods—not every internal helper. Arrows show what each object creates, returns, or contains.

```mermaid
classDiagram
    direction TB

    class Client {
        login(username, password)
        metrics() dict
        query_media() dict
        create_experiment() Experiment
        assignments(experiment_id) Assignments
        logout()
    }
    class Experiment {
        str name
        str seed
        Selection selection
        assign() Assignments
    }
    class Selection {
        tuple filters
        str content_query
        str corpus
        str version
        Optional~str~ search_version
    }
    class MetricFilter {
        str metric_id
        FilterOperator operator
        Scalar value
    }
    class Assignments {
        str experiment_id
        tuple subject_ids
        summary() list~SubjectSummary~
        subject(subject_id) Session
    }
    class SubjectSummary {
        <<TypedDict>>
        str subject
        int trials
        int unique_videos
        int foils
        int cut
        int no_cut
        int cuts_unknown
    }
    class Session {
        Path journal_path
        tuple trials
        tuple pending_trials
        tuple incomplete_trials
        download(destination) tuple~Trial~
        started(trial)
        completed(trial)
        sync() Progress
    }
    class Trial {
        str trial_id
        str media_id
        MediaType media_type
        int block_index
        TrialRole role
        Optional~Segment~ segment
        Optional~Path~ local_path
    }
    class Segment {
        float start_seconds
        float end_seconds
    }
    class Progress {
        str manifest_sha256
        dict trials
    }
    class TrialProgress {
        TrialStatus status
        str attempt_id
    }

    Client ..> Experiment : prepares recipe
    Client ..> Assignments : reopens saved study
    Experiment --> Selection : contains
    Selection --> MetricFilter : zero or more
    Experiment ..> Assignments : assign saves and publishes
    Assignments ..> SubjectSummary : summary returns
    Assignments ..> Session : subject opens
    Session --> Trial : fixed ordered trials
    Trial --> Segment : optional bounds
    Session ..> Progress : sync returns
    Progress --> TrialProgress : keyed by trial ID
```

**Types:** `FilterOperator` = `gt / gte / lt / lte / eq / is / is_not`; `Scalar` here means `int | float | bool | None`. `TrialRole` = `parent / repeat / foil`; `TrialStatus` = `started / completed`. Scalar, TrialRole, and TrialStatus are diagram shorthand, not exported classes. `Optional` means the value may be `None`.

**Media:** `Video`, `Image`, and `Stimulus` inherit from `Media`; only videos currently execute on the server. A `Trial` references media by ID rather than holding a `Video` object. `ExperimentSpec` holds assignment settings internally.

**Boundary:** Client requests go to the website. Session keeps a local SQLite journal and syncs events to the website. PGL/PsychoPy handles playback; the API does not.

If your notebook viewer shows the Mermaid source instead of a figure, use a Mermaid-capable Markdown renderer.

## 1. Setup

Use Python 3.11–3.12. From this directory:

```sh
uv sync
uv run jupyter lab api-walkthrough.ipynb
```

In VS Code, select `notebooks/.venv/bin/python`. The pinned API client is already installed by `uv sync`. A running DBP website with `/api/v1` support and your account is required.

In [ ]:
import sys

print("Kernel Python:", sys.executable)
assert sys.version_info >= (3, 11), "Select a Python 3.11+ kernel."


In [ ]:
from dbp_api import Client, MetricFilter
print("API client ready")


### Imports
Run from here after restarting the kernel.

In [ ]:
from getpass import getpass
from pathlib import Path
from pprint import pprint

from IPython.display import Video as VideoPreview, display
from dbp_api import Client, MetricFilter


## 2. Client: connect to the website

`login()` signs in. `metrics()` lists available metadata. `query_media()` searches. Only your password is prompted; never save it in a cell.

In [ ]:
website_url = "http://127.0.0.1:8773"
username = "your-username"

client = Client(website_url, timeout=120)
client.login(username, getpass("Website password: "))
print("Connected to", website_url)


## 3. Inspect the metadata

Look at metric definitions and coverage before filtering. Missing measurements are not zero or absence.

In [ ]:
metrics = client.metrics()
pprint(metrics)


## 4. Filter without downloading

This example asks for videos at least 10 seconds long. Add a content query such as `hands`, or leave it empty. `corpus` chooses `captions`, `transcripts`, or `both`. Filters are combined with AND.

The response contains server results for this query; `limit=5` limits the returned page, not the study size.

In [ ]:
filters = [MetricFilter("duration_seconds", "gte", 10)]
content_query = ""
corpus = "both"

results = client.query_media(
    filters=filters,
    content_query=content_query,
    corpus=corpus,
    limit=5,
)
pprint(results)


## 5. Experiment → Assignments: create or reopen

`create_experiment()` prepares a local recipe pinned to the dataset/search version. `assign()` asks the server to sample, save, and publish subject assignments.

To create a small demo, set `create_new = True`. To reopen, paste its saved ID and leave `create_new = False`. **Each new assignment call creates a new record.** The seed controls reproducible sampling; reopening an ID retrieves the actual saved assignments.

In [ ]:
experiment_id = ""  # Paste a saved experiment ID to reopen it.
create_new = False  # Change to True only to create a new demo.
seed = "api-walkthrough-1"


In [ ]:
if experiment_id:
    assignments = client.assignments(experiment_id)
else:
    if not create_new:
        raise ValueError("Paste an experiment ID above, or set create_new = True.")
    experiment = client.create_experiment(
        name="API walkthrough",
        seed=seed,
        filters=filters,
        content_query=content_query,
        corpus=corpus,
    )
    assignments = experiment.assign(subjects=2, items_per_subject=5)
    experiment_id = assignments.experiment_id
    create_new = False

print("Save this experiment ID:", experiment_id)
print("Subjects:", assignments.subject_ids)
pprint(assignments.summary())


## 6. Session: choose one subject

`subject()` fetches the fixed trial manifest and creates a local progress journal under `.dbp/`. It does not resample or play anything. `trials` shows the assigned order.

In [ ]:
subject_id = assignments.subject_ids[0]
session = assignments.subject(subject_id, workspace=".dbp")

print("Subject:", subject_id)
print("Progress journal:", session.journal_path)
for trial in session.trials:
    print(trial.trial_id, trial.media_id, trial.role, trial.segment)


## 7. Download and preview

`download()` fetches the assigned files and verifies their hashes. Use a **new destination**; if this session already has downloaded paths, reuse them. This cell transfers media only when you run it.

In [ ]:
destination = Path("api-demo-downloads") / experiment_id / subject_id

if all(trial.local_path is not None for trial in session.trials):
    trials = session.trials
    print("Using previously downloaded files.")
else:
    trials = session.download(destination)

print("First file:", trials[0].local_path)


In [ ]:
trial = trials[0]
display(VideoPreview(filename=str(trial.local_path), embed=True, width=480))


## 8. Progress: what has actually been presented?

`sync()` uploads local events and reads server progress. `pending_trials` contains only unstarted trials. `incomplete_trials` contains started-but-unfinished trials that need review; they are not silently replayed.

Previewing above does not change progress. Another device sees only synchronized events.

In [ ]:
progress = session.sync()
pprint(progress)
print("Unstarted:", len(session.pending_trials))
print("Needs review:", len(session.incomplete_trials))


### Connect your presentation task

This is a template, not an executed cell. Replace `present_and_wait` with your PGL/PsychoPy code. Do not report completion for a cancelled or failed presentation.

```python
for trial in session.pending_trials:
    session.started(trial)  # If this fails, stop before playing.
    present_and_wait(trial.local_path)  # Your task; return only after successful playback.
    session.completed(trial)
```

These API calls perform network/file I/O; keep them outside timing-critical rendering. The presentation program handles timing, responses, and task-result files. Keep `.dbp/` until synchronization is confirmed.

## Optional: foils and shared videos

`assign()` also accepts `foils_per_block`, `shared_per_subject`, `repeats_per_subject`, and `balance_cuts`. Originals are counted by `items_per_subject`; foils and repeats are additional trials.

Foils use the unseen complementary half of a parent whose other half is presented earlier—not a second presentation of an already-seen full video. Segment/cut-balanced studies require the server's prepared segment measurements. Inspect `assignments.summary()` and each trial's `segment` rather than assuming parent metrics describe the half.

This basic demo uses full videos only. Images are not yet supported by the server.

## 9. Finish

Signing out does not delete saved experiments or local files. Clear notebook outputs before sharing: they may contain account data, study IDs, and embedded videos.

In [ ]:
client.logout()
client.close()
